# CV search with implied skills

A recruiter types "Looking for the C++ experts". Keyword search returns the CVs that say C++. It misses the people who
wrote "Qt, STL, LLVM, embedded Linux" and never typed the two characters, and it can't tell that a request for LLVM
experience is close to a request for C++. This notebook builds a search that can, from a collection of CVs alone:

1. **CVs**: 210k real, anonymized IT CVs from Djinni ([dataset](https://huggingface.co/datasets/lang-uk/recruitment-dataset-candidate-profiles-english), MIT; [paper](https://unlp.org.ua/wp-content/uploads/2024/06/drushchak-romanyshyn.pdf)). Skills sit inside the text as written.
2. **Keywords**: mined from the CVs' own skill lists. No outside vocabulary.
3. **Implication graph**: "whoever names A also names B" from co-occurrence across CVs. No labels, no taxonomy, no LLM.
4. **Scorers**: cosine, the graph itself, and order / box / hyperbolic embeddings trained on the graph.
5. **Evaluation**: on a ground truth the CVs carry but nothing above reads, the *Primary Keyword* each candidate picked on the platform.
6. **Search** with an up / down switch, the CVs that name the requirement first: the HR demo, also exported as a standalone page (section 8).

Nothing here reads the earlier skill graph (`data/graph.json`), its labels or its models. Only the method code is shared: the heads and trainers in `skillmatch/methods` and `skillmatch/training.py`.

Run from the repo root. The first run downloads the CVs (226 MB parquet) to `data/djinni/`. The whole notebook takes about 15 minutes on a laptop, most of it training the three embedding models. One seed (0); MPS training isn't bit-reproducible, so a rerun can move the trained rows a little. Every claim in the text cells is computed in the same run.

In [1]:
import html
import os
import re
import time
import warnings

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
warnings.filterwarnings("ignore")

import numpy as np
import torch
from IPython.display import HTML, Markdown, display

from cvsearch import (HEADS, NAMES, TECH, CVIndex, Cooccurrence, KeywordGraph, Search, Vocabulary, column, embed_keywords,
                      eval_cvs, graph_cvs, grid, scorers, subsumption_edges, targets, train, tune)
from cvsearch.demo import scrub  # some CVs still carry profile URLs and phone numbers
from skillmatch.methods import cosine_generality, hybrid

def table(header, rows):
    """A markdown table."""
    fmt = lambda x: f"{x:.3f}" if isinstance(x, float) else str(x)
    return "\n".join(["| " + " | ".join(header) + " |", "|" + "---|" * len(header)] +
                     ["| " + " | ".join(fmt(x) for x in r) + " |" for r in rows])
t0 = time.time()

## 1. The CVs

Each CV is a candidate profile: a title (Position), free text about their experience (Moreinfo) and highlights. A fourth field, "Looking For", holds wishes like "friendly team", so it's left out.

Splits are by a hash of the CV id:
- **graph split (80%)**: the vocabulary and the implication graph are built from these CVs, whatever their Primary Keyword.
- **val (5%) and test (15%)**: CVs whose Primary Keyword is one of 18 technologies, at most 200 / 600 per technology so that JavaScript doesn't swamp Rust.

The **Primary Keyword** is the candidate's own pick from a fixed list on the platform, separate from the text they wrote. It's the ground truth below and nothing upstream of the evaluation reads it. It's noisy (one label per person, self-chosen), and it says "this is their main stack", which is the right truth for a request like "C++ experts".

In [2]:
graph_rows = graph_cvs()                       # [(body, text)]: body = Moreinfo + Highlights, text = Position + body
val_cvs, test_cvs = eval_cvs("val"), eval_cvs("test")
print(f"{len(graph_rows):,} CVs build the graph; {len(val_cvs):,} val and {len(test_cvs):,} test CVs carry the truth")
example = next((c for c in test_cvs if c["pk"] == "C++" and re.search("llvm", c["text"], re.I)), test_cvs[0])
print(f"\nA test CV, Primary Keyword {example['pk']}:\n")
print(scrub(example["text"][:900]))

168,259 CVs build the graph; 2,255 val and 6,713 test CVs carry the truth

A test CV, Primary Keyword C++:

Intern / Trainee C++ Software Developer
Stack: C++ 11 / 14 / 17, Objective-C/Swift, Python, Bash, ASM, JSON, STL, Boost, LLVM/Clang, GCC, Git, CMake, Docker, GNU/Linux (Debian, CentOS), DirectX, OpenGL, Vulkan, OOP/OOD, Data-Oriented & Data-Driven Designs, Design Patterns: Flyweight, Observer, Chain of Responsibilities, Factory, Command, Strategy, Mixin, Thread-safe Singleton, C++'s Policy, CRTP, Pimpl, RAII and SFINAE.
Experience:
Developed iOS apps: CityButler, Flux IoT, TikkR, BubblePerk (as Junior iOS Developer at WallTree);
Term papers and diplomas;
Custom graphical framework implementation;
Experience with popular game engines (Unity3D, UE4).


## 2. Keywords from the CVs

CVs list their skills: "Strong C, C++, bash, ARM TrustZone, gcc, llvm, clang". The vocabulary is what those lists repeat (`Vocabulary.mine`):

- **A list item** is 1–3 tokens and at most 30 characters, inside a run of at least three such items. Prose split by commas rarely produces three short pieces in a row, so "Currently, I work at …" doesn't count.
- **A keyword** is listed by at least 5 CVs. The floor is that low on purpose: LLVM is listed by exactly 5.
- **Lower-case words** ("using", "currently") must sit in lists in at least 20% of the CVs that use them; those are mostly prose.
- **Spellings merge**: react.js = reactjs = React JS, and a version folds into its keyword (C++17 → C++, Python3 → Python, .NET 6 → .NET).
- **Extraction** (`Vocabulary.extract`) finds every keyword span, so "Java Developer" names both Java Developer and Java. A one-word keyword written with spaces names only itself: "Java Script" is JavaScript, not also Java. Keywords of one or two letters (C, R, Go) count only when not written in lower case, so "ready to go" isn't Go.

In [3]:
vocab = Vocabulary.mine(body for body, _ in graph_rows)
print(f"{len(vocab):,} keywords, {len(vocab.variants)} version spellings folded into them")
by_df = sorted(range(len(vocab)), key=lambda i: -vocab.df[i])
print("most listed:", ", ".join(vocab[i] for i in by_df[:40]))
print("around rank 3000:", ", ".join(vocab[i] for i in by_df[3000:3025]))
print("the long tail:", ", ".join(vocab[i] for i in by_df[-25:]))

def highlight(text, names, width=None):
    """The CV text with the named keywords marked."""
    out = html.escape(text if width is None else text[:width])
    for n in sorted(set(names), key=len, reverse=True):
        out = re.sub(r"(?<![\w+#])(" + re.escape(html.escape(n)) + r")(?![\w+#])", r"<mark>\1</mark>", out, flags=re.I)
    return out
found = [vocab[i] for i in vocab.extract(example["text"])]
print(f"\nThe example CV names {len(found)} keywords:")
display(HTML(f"<div style='font-size:13px;line-height:1.5'>{highlight(scrub(example['text']), found, 900)}</div>"))

12,521 keywords, 331 version spellings folded into them
most listed: CSS, Git, JavaScript, HTML, Jira, MySQL, SQL, Docker, React, PostgreSQL, TypeScript, Node.js, Python, MongoDB, Java, Redux, jQuery, JS, Postman, Linux, SASS, Jenkins, PHP, Bootstrap, REST API, Figma, Confluence, AWS, Redis, Scrum, C#, SCSS, UI, Trello, GitHub, Maven, JSON, React.js, XML, Agile
around rank 3000: Daily Scrum meetings, Dataflow, data preprocessing, DBUnit, DevExtreme, Display, Embedded C, Engineering Manager, England, Entity, ERP system, Experience with JavaScript, Fortigate, Front End Dev, Ganache, GetResponse, GitHub Action, Google Documents, Google Play Console, Google Test, Gorm, GSM, Head of Sales, Hunter, Jinja
the long tail: XRumer, Yahoo Ads, Yalantis, YandexMapKit, Yandex.Money, Yandex Wordstat, Yocto Project, Yotpo, YouScan, YT, Yves Rocher, Zabix, Zap, Zappier, ZBrash, Zbrush sculpting, Zebroid, Zen Cart, Zend Studio, Zoho Recruit, Zookeper, Zopim, сandidate pitching, СJM, Сommunication

The e

## 3. The implication graph

The API: `Cooccurrence.count` over the keyword sets of the graph-split CVs, then `subsumption_edges` and `KeywordGraph`. `build_graph(cvs)` does sections 2 and 3 in one call.

**The rule** (subsumption, Sanderson & Croft 1999, adapted): A → B, "whoever has A has B", when
- **P(B | A) ≥ 0.5**: most CVs naming A also name B;
- **n(B) ≥ 1.5 · n(A)**: B is clearly more common. Near-equivalents (HTML and CSS) get no edge, and the graph can't have cycles;
- **lift ≥ 3**: P(B | A) is at least three times B's base rate. So anything named in a third of all CVs or more (Developer, Git) can't be a target, and hubs don't absorb everything;
- **n(A, B) ≥ 5**;
- **B isn't part of A's own name**: "Unreal Engine" and "Engine" always co-occur, by construction.

No labels, no taxonomy, no LLM. `graph.verify` with `nli_judge()` (a local NLI model) can filter the edges by direction, but it isn't run here. That's about 100k NLI calls, and whether it helps is untested.

In [4]:
t = time.time()
keyword_sets = [vocab.extract(text) for _, text in graph_rows]
co = Cooccurrence.count(keyword_sets, len(vocab))
contained = vocab.contained()
X = embed_keywords(vocab)                      # bge-base on the names: the trained heads and cosine read these
graph = KeywordGraph(vocab, subsumption_edges(co, contained), X)
print(f"built in {time.time() - t:.0f}s; mean {np.mean([len(s) for s in keyword_sets]):.1f} keywords per CV")
print(graph.stats())
print("\nHow the threshold on P(B | A) trades edges for confidence:")
display(Markdown(table(["P(B | A) ≥", "edges", "implied pairs (training positives)"],
    [[tau, len(e := subsumption_edges(co, contained, tau=tau)), KeywordGraph(vocab, e, X[:, :1]).stats()["closure pairs"]]
     for tau in (0.4, 0.5, 0.6, 0.7)])))

built in 41s; mean 32.8 keywords per CV
{'keywords': 12521, 'keywords with an edge': 9488, 'edges': 47929, 'closure pairs': 55906, 'longest chain': 7}

How the threshold on P(B | A) trades edges for confidence:


| P(B | A) ≥ | edges | implied pairs (training positives) |
|---|---|---|
| 0.400 | 74527 | 100367 |
| 0.500 | 47929 | 55906 |
| 0.600 | 28112 | 30669 |
| 0.700 | 16524 | 17385 |

In [5]:
rows = []
for n in ["LLVM", "Clang", "Qt", "STL", "Spring Boot", "Hibernate", "Redux", "Django", "Laravel", "Kubernetes", "SwiftUI", "Jetpack Compose"]:
    rows.append([n, ", ".join(f"{b} ({p:.2f})" for b, p in graph.implies(n)[:5]) or "–"])
display(Markdown("**What having a keyword implies** (direct edges, P(B | A)):\n\n" + table(["keyword", "implies"], rows)))
inc = graph.implied_by("C++")
display(Markdown(f"**{len(inc)} keywords imply C++.** The most certain 30: " + ", ".join(f"{a} ({p:.2f})" for a, p in inc[:30])))
rng = np.random.default_rng(0)
sample = [list(graph.G.edges)[i] for i in rng.choice(graph.G.number_of_edges(), 20, replace=False)]
display(Markdown("**20 edges at random**, to eyeball the noise: " + "; ".join(f"{vocab[a]} → {vocab[b]}" for a, b in sample)))

**What having a keyword implies** (direct edges, P(B | A)):

| keyword | implies |
|---|---|
| LLVM | C++ (0.95), Clang (0.55) |
| Clang | C++ (0.94), Linux (0.62), GCC (0.50), Python (0.50) |
| Qt | C++ (0.91) |
| STL | C++ (0.97) |
| Spring Boot | Java (0.97), Java Developer (0.64) |
| Hibernate | Java (0.99), Spring (0.95), Java Developer (0.72) |
| Redux | Frontend (0.79), React (0.78), Frontend Developer (0.61), JavaScript (0.54), CSS (0.50) |
| Django | Python (0.92) |
| Laravel | PHP (0.83) |
| Kubernetes | Docker (0.70), AWS (0.54) |
| SwiftUI | iOS (0.94), iOS Developer (0.78), Swift (0.65), UIKit (0.54) |
| Jetpack Compose | Android (0.97), Android Developer (0.84), Kotlin (0.81), MVVM (0.60), Java (0.57) |

**178 keywords imply C++.** The most certain 30: 3D math (1.00), Boost.Asio (1.00), cpprestsdk (1.00), GLFW (1.00), Google Mock (1.00), POSIX Threads (1.00), qmake (1.00), Qt Widgets (1.00), vcpkg (1.00), VTK (1.00), GMock (0.97), OpenCL (0.97), STL (0.97), GTest (0.96), Qt Creator (0.96), CMake (0.95), LLVM (0.95), Qt framework (0.95), SFML (0.95), Clang (0.94), WTL (0.94), wxWidgets (0.94), Google Test (0.93), MFC (0.93), Qt Developer (0.93), Cocos2d-x (0.92), Eigen (0.92), QML (0.92), Emscripten (0.91), libcurl (0.91)

**20 edges at random**, to eyeball the noise: Unity IAP → Android; firewalld → DevOps; Error Guessing → Bug; Visualforce pages → Apex; Mantine UI → Material UI; ORMLite → Retrofit; Mule ESB → Spring; Working with database → QA Engineer; PHP 7+ → MySQL; Positive-Negative → Bug; contract administration → change management; Sprint Retrospective → sprint planning; Phing → Symfony; Adobe Premier → Premier Pro; Moleculer → Full Stack; AWS Batch → Lead; TCA → Swift; Lucidchart → Jira; Retina support → AJAX; Antd → Frontend Developer

## 4. The scorers

Every method scores s(A → B), "having keyword A implies having keyword B", from keyword names alone:

| method | what it is | trained |
|---|---|---|
| cosine | cos of the bge-base vectors of the two names. Symmetric. | no |
| graph lookup | the graph: 2 if A = B, 1 + P along the best path if the graph implies it, else 0; 0.01·cosine orders the rest | no |
| cos + gen | cosine + λ·generality(B), generality = log(1 + #keywords that imply B) | no |
| order + gen + cos | order embedding (Vendrov+ 2016) + λ·gen + μ·cos | on the graph |
| box + gen + cos | Gumbel box embedding (Dasgupta+ 2020) + λ·gen + μ·cos | on the graph |
| hyperbolic + gen + cos + depth | Poincaré embedding (Nickel & Kiela 2017) + λ·gen + μ·cos + μ_d·d(0, A), the depth term for the down direction | on the graph |

The heads train on the graph's implied pairs with negatives (reversed pairs, random keywords, cousins). Their configs are the methods' defaults. λ, μ and μ_d are picked on **val**, for the mean of down-unseen MAP and up-unseen MRR (defined in section 5).

In [6]:
heads = {}
for h in HEADS:
    t = time.time()
    heads[h] = train(graph, h, epochs=100)
    print(f"{NAMES[h]:32} trained in {time.time() - t:.0f}s on {len(graph.positives):,} implied pairs")
val = CVIndex(val_cvs, vocab)
LM = [(l, m) for l in (0, 0.1, 0.3, 1) for m in (0, 1, 3, 10)]
w = {"cosgen": tune(val, lambda o: cosine_generality(graph.X, graph.generality, *o), [(l,) for l in (0, 0.01, 0.03, 0.1, 0.3, 1)])[0]}
for h in ("order", "box"):
    w[h] = tune(val, lambda o, h=h: hybrid(heads[h].scorer(graph.X), graph, *o), LM)[0]
def with_depth(o):
    heads["hyp"].mu = o[2]
    return hybrid(heads["hyp"].scorer(graph.X), graph, *o[:2])
w["hyp"] = tune(val, with_depth, [(l, m, d) for l, m in LM for d in (0, 0.5, 1, 1.5)])[0]
S = scorers(graph, heads, w)
print("\nweights picked on val:", {NAMES[k]: v for k, v in w.items()})

order + gen + cos                trained in 149s on 55,906 implied pairs


box + gen + cos                  trained in 190s on 55,906 implied pairs


hyperbolic + gen + cos + depth   trained in 197s on 55,906 implied pairs



weights picked on val: {'cos + gen': (0,), 'order + gen + cos': (0, 1), 'box + gen + cos': (0, 1), 'hyperbolic + gen + cos + depth': (0, 10, 1)}


## 5. The evaluation grid (test)

The truth is the Primary Keyword p, for the 18 technologies. "Names p" means the CV text names p's keyword. Aliases count: JS for JavaScript, Go for Golang, node for Node.js, versions like C++17.

| query type | the question | candidates | relevant | metric |
|---|---|---|---|---|
| **down seen** | "who are the C++ people?", among CVs that say C++ | test CVs naming p | Primary Keyword p | MAP, averaged over p |
| **down unseen** | the same, among CVs that never say C++ | test CVs not naming p | Primary Keyword p | MAP |
| **up seen** | "what is this person's main stack?", for a CV that names it | the 18 keywords | its own p | MRR, accuracy@1 |
| **up unseen** | the same, for a CV that never names its own stack | the 18 keywords | its own p | MRR, accuracy@1 |
| **identity** | does a keyword come first for itself? | all keywords | the keyword itself | share at rank 1 (keyword level); AUC of CVs naming p over the rest (CV level) |

A CV's score toward p is the best score of its keywords toward p's keyword. **Keyword search** is the baseline: score 1 if the CV names p, else 0, with ties broken at random. It is perfect at finding who names p, and blind to everyone else. Random ranking scores the share of relevant CVs.

How often the text doesn't name the candidate's own Primary Keyword, the people "down unseen" is about:

In [7]:
test = CVIndex(test_cvs, vocab)
T = targets(vocab)
pk = np.array([c["pk"] for c in test_cvs])
display(Markdown(table(["Primary Keyword", "test CVs", "text doesn't name it"],
    [[p, int((pk == p).sum()), f"{(~test.names(T[p][1]).numpy()[pk == p]).mean():.0%}"] for p in TECH])))

| Primary Keyword | test CVs | text doesn't name it |
|---|---|---|
| JavaScript | 600 | 39% |
| Java | 600 | 6% |
| .NET | 600 | 18% |
| Python | 600 | 7% |
| PHP | 600 | 12% |
| Node.js | 600 | 18% |
| DevOps | 594 | 6% |
| iOS | 468 | 2% |
| Android | 486 | 2% |
| C++ | 434 | 18% |
| Unity | 268 | 42% |
| Ruby | 181 | 4% |
| Golang | 202 | 12% |
| SQL | 176 | 21% |
| Flutter | 172 | 3% |
| Scala | 55 | 9% |
| Rust | 25 | 32% |
| Salesforce | 52 | 12% |

In [8]:
t = time.time()
results = {"keyword search": grid(test, None)}
for k, M in S.items(): results[NAMES[k]] = grid(test, M)
cols = ["down seen MAP", "down unseen MAP", "up seen MRR", "up seen acc@1", "up unseen MRR", "up unseen acc@1", "identity: keyword top 1", "identity: CV AUC"]
display(Markdown(table(["method"] + cols, [[k] + [r.get(c, "–") for c in cols] for k, r in results.items()])))
chance = np.mean([np.mean(pk[~test.names(T[p][1]).numpy()] == p) for p in TECH])
print(f"scored in {time.time() - t:.0f}s; random ranking on down unseen ≈ {chance:.3f}")

| method | down seen MAP | down unseen MAP | up seen MRR | up seen acc@1 | up unseen MRR | up unseen acc@1 | identity: keyword top 1 | identity: CV AUC |
|---|---|---|---|---|---|---|---|---|
| keyword search | 0.565 | 0.010 | 0.812 | 0.674 | 0.176 | 0.041 | – | – |
| cosine | 0.686 | 0.057 | 0.857 | 0.751 | 0.316 | 0.162 | 1.000 | 1.000 |
| graph lookup | 0.781 | 0.165 | 0.874 | 0.783 | 0.439 | 0.287 | 1.000 | 0.998 |
| cos + gen | 0.686 | 0.057 | 0.857 | 0.751 | 0.316 | 0.162 | 1.000 | 1.000 |
| order + gen + cos | 0.702 | 0.064 | 0.838 | 0.719 | 0.329 | 0.175 | 1.000 | 1.000 |
| box + gen + cos | 0.666 | 0.070 | 0.653 | 0.507 | 0.342 | 0.183 | 0.537 | 0.977 |
| hyperbolic + gen + cos + depth | 0.750 | 0.117 | 0.920 | 0.859 | 0.341 | 0.186 | 1.000 | 0.999 |

scored in 78s; random ranking on down unseen ≈ 0.009


In [9]:
R = results
scored = [k for k in R if k != "keyword search"]
quality = cols[:6]
best = lambda c: max(scored, key=lambda k: R[k][c])
g, kw, cos = R["graph lookup"], R["keyword search"], R["cosine"]
wins = [c for c in quality if best(c) == "graph lookup"]
trained = [NAMES[k] for k in ("order", "box", "hyp")]
beat = [(c, best(c)) for c in quality if best(c) in trained]
zero = [NAMES[k] for k, v in w.items() if v[0] == 0]
weak = [k for k in scored if R[k]["identity: keyword top 1"] < 0.9]
display(Markdown(f"""
**What the grid says**

- **Keyword search is blind to the unseen half.** Down unseen {kw['down unseen MAP']:.3f} is chance level ({chance:.3f}). The best scorer there is {best('down unseen MAP')} with {R[best('down unseen MAP')]['down unseen MAP']:.3f}, about {R[best('down unseen MAP')]['down unseen MAP'] / chance:.0f}× chance. Finding a C++ person who never says C++ is hard, and the numbers are low in absolute terms.
- **The graph itself, with no training, is the best scorer on {len(wins)} of {len(quality)} columns**: {g['down seen MAP']:.3f} / {g['down unseen MAP']:.3f} down MAP and {g['up seen acc@1']:.3f} / {g['up unseen acc@1']:.3f} up accuracy@1, against cosine's {cos['down seen MAP']:.3f} / {cos['down unseen MAP']:.3f} and {cos['up seen acc@1']:.3f} / {cos['up unseen acc@1']:.3f}. {"A trained model is best on " + "; ".join(f"{c} ({k}, {R[k][c]:.3f} vs graph {g[c]:.3f})" for c, k in beat) + "." if beat else "No trained model beats it on any column."} The trained embeddings learn from this graph, and every test keyword is in the same closed vocabulary, so they have little to generalize to. Where they could matter, keywords the graph has no edge for, is untested.
- **Generality:** val picked λ = 0 for {", ".join(zero) if zero else "no method"}{" (so cos + gen ranks exactly like cosine)" if "cos + gen" in zero else ""}.
- **Identity:** {(", ".join(f"{k} ({R[k]['identity: keyword top 1']:.2f})" for k in weak) + " doesn't rank every keyword first for itself. That doesn't hurt search, because search lists the CVs that name the requirement first anyway (section 6).") if weak else "every scorer ranks each keyword first for itself, and search lists the CVs that name the requirement first anyway (section 6)."}
"""))


**What the grid says**

- **Keyword search is blind to the unseen half.** Down unseen 0.010 is chance level (0.009). The best scorer there is graph lookup with 0.165, about 18× chance. Finding a C++ person who never says C++ is hard, and the numbers are low in absolute terms.
- **The graph itself, with no training, is the best scorer on 4 of 6 columns**: 0.781 / 0.165 down MAP and 0.783 / 0.287 up accuracy@1, against cosine's 0.686 / 0.057 and 0.751 / 0.162. A trained model is best on up seen MRR (hyperbolic + gen + cos + depth, 0.920 vs graph 0.874); up seen acc@1 (hyperbolic + gen + cos + depth, 0.859 vs graph 0.783). The trained embeddings learn from this graph, and every test keyword is in the same closed vocabulary, so they have little to generalize to. Where they could matter, keywords the graph has no edge for, is untested.
- **Generality:** val picked λ = 0 for cos + gen, order + gen + cos, box + gen + cos, hyperbolic + gen + cos + depth (so cos + gen ranks exactly like cosine).
- **Identity:** box + gen + cos (0.54) doesn't rank every keyword first for itself. That doesn't hurt search, because search lists the CVs that name the requirement first anyway (section 6).


## 6. Search, and the HR demo

`Search(index, scorers).query(text, mode, method)` reads the keywords a request names (no LLM), **longest match first**: "Need a Spring Boot developer" is Spring Boot, not also Spring and Boot. Keywords that a third or more of the CVs name, like "Developer", match too many CVs to be a requirement and are dropped. That's the same bound that keeps them from being graph targets. Then it ranks the CVs:

1. **CVs that name the requirement come first**, whatever the mode. This is the identity correspondence: a CV that says LLVM answers "LLVM" before any CV that only implies it.
2. **The rest are a fallback**, scored by their best keyword plus 0.1 × the mean of their three best, so more support counts:
   - **down**, "who has it": s(CV keyword → requirement). Qt, STL and LLVM people for "C++".
   - **up**, "what it implies": s(requirement → CV keyword), plus the down score. A C++ CV is the closest thing to an LLVM CV, and among C++ CVs the ones that also look like LLVM people (Clang, GCC) come first, not every CV that mentions C++ once.

The search runs over the 6.7k test CVs. The Primary Keyword column is the truth, shown only to check the results.

In [10]:
search = Search(test, S)

def show(text, mode, method, named=3, implied=8, expect=None):
    req, top = search.query(text, mode, method, k=named, names=True)
    _, rest = search.query(text, mode, method, k=implied, names=False)
    rows = []
    for i, r in enumerate(top + rest, 1):
        c, why = {**r["cv"], "text": scrub(r["cv"]["text"])}, r["because"][0]
        how = f"names {', '.join(req)}" if r["names"] else f"via <b>{html.escape(why or '–')}</b>"
        hit = "✓" if expect and c["pk"] == expect else ""
        m = re.search(re.escape(why or ""), c["text"], re.I) if why else None
        lo = max(0, m.start() - 70) if m else 0
        rows.append(f"<tr><td>{i}</td><td>{how}</td><td>{html.escape(c['position'][:45])}</td><td>{c['pk']} {hit}</td>"
                    f"<td style='font-size:12px'>…{highlight(c['text'][lo:lo + 180], [why] if why else [])}…</td></tr>")
    hits = sum(r["cv"]["pk"] == expect for r in rest) if expect else None
    head = f"<b>{NAMES[method]}</b>, {mode}, requirement: {', '.join(req) or 'none found'}" + \
           (f" — {hits}/{len(rest)} of the fallback have Primary Keyword {expect}" if expect else "")
    display(HTML(f"<p>{head}</p><table><tr><th>#</th><th>why</th><th>title</th><th>Primary Keyword</th><th>where</th></tr>{''.join(rows)}</table>"))

### "Would be nice if has LLVM knowledge" (up)

LLVM is rare, so only a few test CVs name it. They come first; after them comes the fallback, CVs with what LLVM implies.

In [11]:
for m in ("graph", "box", "hyp", "cos"):
    show("Would be nice if has LLVM knowledge", "up", m, named=4, implied=6, expect="C++")

#,why,title,Primary Keyword,where
1,names LLVM,Intern / Trainee C++ Software Developer,C++ ✓,"…11 / 14 / 17, Objective-C/Swift, Python, Bash, ASM, JSON, STL, Boost, LLVM/Clang, GCC, Git, CMake, Docker, GNU/Linux (Debian, CentOS), DirectX, OpenGL, Vulkan, OOP/OOD, Data-Orient…"
2,names LLVM,C++ Developer,C++ ✓,"…ction and debugging tools • CMake/make, python/bash, docker • Using LLVM utils for code formatting and linting, profiling/trace utils, gdb/ldb • BS in Computer Science. …"
3,names LLVM,Senior Software Engineer,C++ ✓,"…ule for fast executable (Mach-O/ELF) files parsing, created own LLDB (LLVM debugger) fork with fixes to allow mini core dumps debugging and iOS dumps cross-debugging on Linux. W…"
4,names LLVM,Senior C++ Developer,C++ ✓,"…ipt, Java, Bash, AOP Libraries & Framework: boost, Qt, wxWidgets, LLVM, curl, icu. Familiar with Spring Framework Tools & Technologies: Git, SVN, Visual Studio, Eclipse, F…"
5,via C++,Lead C++ / Performance Analyst,C++ ✓,"…Lead C++ / Performance Analyst Performance analysis, debugging and reverse-engineering (publication at RSDN Magazine); Computer architecture: CPU internals, caches, memory, code …"
6,via C++,Junior-Middle C/C++ Developer,C++ ✓,…Junior-Middle C/C++ Developer C/C++ developer with experience in working on various personal projects and on freelanсe( C about - +3 year/C++ - half year). One and half year of co…
7,via C++,Embedded developer,C++ ✓,"…Embedded developer C++, C, Assembler, ARM_NEON, AVX, SSE, SIMD, MIMD Increase performance in 3-4 times comparing to Clang (Release mode)…"
8,via C++,Golang Developer,Golang,"…ry language - .NET Less comfortable: - node.js - JVM stack - C/C++ - ask me, it could be possible that I did something before I worked as prod- and dev- DBA. I have d…"
9,via C++,"Erlang, Haskell, Python, C, C++; systems/dist",C++ ✓,"…Erlang, Haskell, Python, C, C++; systems/distributed programming; Languages: C, C++, Python, Erlang, Haskell, bash/sh. Technologies: docker/vagrant/xen, http/tcp/ip, file systems …"
10,via C++,Software Engineer/Tech Lead/Software Architec,Rust,"… multimedia. I dealt with the following: Programming languages: C/C++ (primary), Rust (primary), HTML, JavaScript, CSS, Python, Lua, m4, PineScript Technologies & tools: Jav…"


#,why,title,Primary Keyword,where
1,names LLVM,Intern / Trainee C++ Software Developer,C++ ✓,"…Bash, ASM, JSON, STL, Boost, LLVM/Clang, GCC, Git, CMake, Docker, GNU/Linux (Debian, CentOS), DirectX, OpenGL, Vulkan, OOP/OOD, Data-Oriented & Data-Driven Designs, Design Patterns…"
2,names LLVM,Senior Software Engineer,C++ ✓,"…el code including direct Mach-O/ELF format reading/writing. Backend: Linux, Python 3.7, PostgreSQL, SQLAlchemy, Alembic, aiohttp, graphql-server, marshmallow Frontend: React, Mat…"
3,names LLVM,Senior C++ Developer,C++ ✓,"… STL, OOP, TDD, Design Patterns, Scrum, VHDL, Verilog. Familiar with JavaScript, Java, Bash, AOP Libraries & Framework: boost, Qt, wxWidgets, LLVM, curl, icu. Familiar with S…"
4,names LLVM,C++ Developer,C++ ✓,"…C++ Developer • C++17 knowledge • Parallelism and multithreading • Linux superuser, strong notion of OS concepts and system API • TCP/IP, HTTP, DNS and other application layer p…"
5,via MVC,Junior .NET / C# Full Stack Developer,.NET,"…oring the developers. Skilled in C# / .NET, ASP .NET Core WebApi / MVC / Blazor / Web Forms, Razor pages, Electron, HTML, CSS / SASS / SCSS, React with TypeScript, Docker, MSSQ…"
6,via Virtualization,C++ Developer,C++ ✓,"…with Go and Bash as a DevOps DevOps skills: • *nix • Virtualization: XenServer, KVM, VMware esxi • Containerization: Docker (compose, Swarm, K8s), LXC • Web Serv…"
7,via Python,Разработчик C++,C++ ✓,"…e, knowledge of modern c++ idioms like sfinae, members detector... Python, django (with sertificate), pandas / numpy / theano / scikit-learn R Language, haskell with some per…"
8,via Linux,Senior/Team Lead in Computer Vision,C++ ✓,"…rastructure: Git and SVN, Bash Shell, CMake, Make Operating systems: Linux (Fedora Core 5+, Open Suse, gentoo, Ubuntu), Windows System administration (9+ years): Active Directory…"
9,via Embedded,C++ Developer,C++ ✓,"… 2018 – Jan 2020 Middle C++ Developer _ Luxoft Automotive project - embedded navigation. Guiding data analysis & rendering (OpenGL ES), processing issues & implementing new featu…"
10,via Linux,Node.js Engineer,Node.js,…inerization technologies such as Docker • Proficient in working with Linux terminals (also VIM editor) • Experience with different CI/CD and version control tools • Experience i…


#,why,title,Primary Keyword,where
1,names LLVM,Intern / Trainee C++ Software Developer,C++ ✓,"…11 / 14 / 17, Objective-C/Swift, Python, Bash, ASM, JSON, STL, Boost, LLVM/Clang, GCC, Git, CMake, Docker, GNU/Linux (Debian, CentOS), DirectX, OpenGL, Vulkan, OOP/OOD, Data-Orient…"
2,names LLVM,C++ Developer,C++ ✓,"…ction and debugging tools • CMake/make, python/bash, docker • Using LLVM utils for code formatting and linting, profiling/trace utils, gdb/ldb • BS in Computer Science. …"
3,names LLVM,Senior C++ Developer,C++ ✓,"…ipt, Java, Bash, AOP Libraries & Framework: boost, Qt, wxWidgets, LLVM, curl, icu. Familiar with Spring Framework Tools & Technologies: Git, SVN, Visual Studio, Eclipse, F…"
4,names LLVM,Senior Software Engineer,C++ ✓,"…ule for fast executable (Mach-O/ELF) files parsing, created own LLDB (LLVM debugger) fork with fixes to allow mini core dumps debugging and iOS dumps cross-debugging on Linux. W…"
5,via Linux,Lead C++ / Performance Analyst,C++ ✓,"…hi [7-XE]; Lotus Notes: LS/C/C++/Java/COM API; Operating systems: Linux, Mac OSX, MS Windows; Task-tracking: Jira, redmine, ClearQuest, Bugzilla; Version control systems: S…"
6,via Linux,Senior C/C++ Developer,C++ ✓,"…enior C/C++ Developer Programming languages: C, С++ ,Python Embedded Linux (User-Space/Kernel), RTOS development, Device drivers development Source control system : Git, Mercuria…"
7,via Linux,"Erlang, Haskell, Python, C, C++; systems/dist",C++ ✓,"… Unix tools, GCC toolchains, GDB, valgrind, GHC, vim. OS, platforms: Linux, POSIX. - algorithms; - parsing, interpreters, compilers, assemblers (x86, mips) knowledge; - soft…"
8,via Linux,C++ Developer,C++ ✓,"… Programming Languages : C++, python, bash Technologies : Embedded Linux, gcc, Cmake, boost, Qt Source Control Tools : Git Databases : sqlite3 Currently I have 5 projects in f…"
9,via Linux,Senior Embedded Developer,C++ ✓,"…,​ ​Java,​ ​Verilog,​ ​ AHDL etc. - FreeRTOS,​ ​scmRTOS,​ Windows,​ ​Linux,​ ​bare​ ​metal​ ​solutions. - Cortex-M0/M3,​ ​MSP430,​ ​AVR,​ ​Freescale,​ MCS-51,​ ​PIC,​ ​Nios​ II​.…"
10,via Linux,Embedded developer (SW/HW),C++ ✓,"… Maple 9.0, MathCAD, Matlab, Origin, ROOT Operation systems: • Linux • Windows …"


#,why,title,Primary Keyword,where
1,names LLVM,Intern / Trainee C++ Software Developer,C++ ✓,"…11 / 14 / 17, Objective-C/Swift, Python, Bash, ASM, JSON, STL, Boost, LLVM/Clang, GCC, Git, CMake, Docker, GNU/Linux (Debian, CentOS), DirectX, OpenGL, Vulkan, OOP/OOD, Data-Orient…"
2,names LLVM,Senior C++ Developer,C++ ✓,"…ipt, Java, Bash, AOP Libraries & Framework: boost, Qt, wxWidgets, LLVM, curl, icu. Familiar with Spring Framework Tools & Technologies: Git, SVN, Visual Studio, Eclipse, F…"
3,names LLVM,C++ Developer,C++ ✓,"…ction and debugging tools • CMake/make, python/bash, docker • Using LLVM utils for code formatting and linting, profiling/trace utils, gdb/ldb • BS in Computer Science. …"
4,names LLVM,Senior Software Engineer,C++ ✓,"…ule for fast executable (Mach-O/ELF) files parsing, created own LLDB (LLVM debugger) fork with fixes to allow mini core dumps debugging and iOS dumps cross-debugging on Linux. W…"
5,via LVM,TechOps Engineer,DevOps,"…upport via helpdesk system Experience with KVM/XEN – node setup with LVM thin pool Basic Virtualization knowledge (KVM, Xen, OpenVZ) Basic knowledge of TCP/IP, juniper and netwo…"
6,via LVM,DevOps engineer,DevOps,"… and troubleshooting: systemd/init.d, journald, ps/df/netstat/iostat, lvm, mdadm. Windows administration and troubleshooting, implementation and administration Terminal Services…"
7,via LVM,DevOps engineer,DevOps,"…al volumes for file system growth needs using Logical Volume Manager (LVM) commands - Experience with enterprise tools such as Ansible,Splunk,Qradar,Spacewalk,Openshift,Nginx WAF,…"
8,via LVM,DevOps engineer,DevOps,"…ministration: Nginx, Git, DNS, NTP, LAMP, OpenVPN, Node, SSH, IPsec, LVM и другие - Scripting: Bash, Yaml, Python - Docker: docker-compose, Docker Swarm, Kubernetes (basic) …"
9,via JVM,Tech Lead / Senior Full-stack Engineer [Node.,Node.js,"…d mobile: Cordova / PhoneGap, Adobe AIR, Android SDK, Flutter Java / JVM: Spring Boot, Spring-security, JavaEE / JPA / Servlets / JSP / JSTL, Groovy / Grails / GORM, Clojure REST…"
10,via nvm,Embedded Software Engineer,C++ ✓,"…, PduR, Autosar Com, CanSM, CANNM) and Diag Stack modules (DCM, DEM, NvM) to know how. • Working on Com Stack bring up activities by importing a sample dbc file into Vector DaVin…"


**Checking the up-mode fallback.** Up mode has no ground truth, so here is a check on 22 hand-picked requests whose fallback people are obvious: Spring → Java people, Django → Python people, SwiftUI → iOS people, and so on (`search.UP_CHECK`). For each request, the share of the first 20 fallback CVs (those that don't name it) whose Primary Keyword is the obvious one. The pairs are my picks, not data.

In [12]:
from cvsearch import UP_CHECK, up_check
checks = {m: up_check(search, m) for m in ("graph", "hyp", "box", "order", "cos")}
asked = list(checks["graph"])
display(Markdown(table(["request → obvious people"] + [NAMES[m] for m in checks],
    [[f"{r} → {dict(UP_CHECK)[r]}"] + [f"{checks[m].get(r, float('nan')):.0%}" for m in checks] for r in asked] +
    [["**mean**"] + [f"**{np.mean(list(checks[m].values())):.0%}**" for m in checks]])))

| request → obvious people | graph lookup | hyperbolic + gen + cos + depth | box + gen + cos | order + gen + cos | cosine |
|---|---|---|---|---|---|
| Spring → Java | 95% | 80% | 55% | 0% | 0% |
| Spring Boot → Java | 95% | 90% | 80% | 85% | 80% |
| Hibernate → Java | 95% | 90% | 75% | 30% | 25% |
| LLVM → C++ | 80% | 55% | 45% | 10% | 10% |
| Qt → C++ | 50% | 60% | 40% | 20% | 20% |
| STL → C++ | 55% | 0% | 5% | 15% | 35% |
| Unreal Engine → C++ | 15% | 20% | 35% | 10% | 20% |
| Django → Python | 85% | 55% | 55% | 60% | 10% |
| Flask → Python | 90% | 70% | 75% | 25% | 25% |
| FastAPI → Python | 90% | 95% | 0% | 5% | 0% |
| Laravel → PHP | 85% | 85% | 80% | 65% | 55% |
| Symfony → PHP | 95% | 75% | 90% | 70% | 0% |
| SwiftUI → iOS | 100% | 100% | 85% | 100% | 95% |
| UIKit → iOS | 100% | 95% | 90% | 90% | 80% |
| Jetpack Compose → Android | 95% | 95% | 55% | 90% | 90% |
| Ruby on Rails → Ruby | 70% | 80% | 30% | 80% | 80% |
| Gin → Golang | 60% | 35% | 5% | 15% | 10% |
| ASP.NET → .NET | 100% | 100% | 85% | 95% | 90% |
| Entity Framework → .NET | 90% | 90% | 85% | 90% | 95% |
| NestJS → Node.js | 65% | 70% | 35% | 85% | 85% |
| Terraform → DevOps | 75% | 75% | 60% | 60% | 55% |
| Ansible → DevOps | 80% | 25% | 50% | 50% | 50% |
| **mean** | **80%** | **70%** | **55%** | **52%** | **46%** |

In [13]:
lines = []
for m in ("graph", "box", "hyp", "cos"):
    _, rest = search.query("Would be nice if has LLVM knowledge", "up", m, k=20, names=False)
    via = [r["because"][0] for r in rest]
    lines.append(f"- **{NAMES[m]}**: the first 20 fallback CVs come via {', '.join(dict.fromkeys(via))}; "
                 f"{sum(r['cv']['pk'] == 'C++' for r in rest)}/20 have Primary Keyword C++.")
display(Markdown("\n".join(lines) + f"""

The graph's edges out of LLVM: {", ".join(f"{b} ({p:.2f})" for b, p in graph.implies("LLVM")) or "none"}. Cosine reads names only, so its fallback follows spelling. A hyperbolic distance is symmetric, so central hubs are close to everything, and its depth term only acts in down mode. There is no ground truth for up-mode fallbacks, so this part is an eyeball check, not a measurement.
"""))

- **graph lookup**: the first 20 fallback CVs come via C++; 16/20 have Primary Keyword C++.
- **box + gen + cos**: the first 20 fallback CVs come via MVC, Virtualization, Python, Linux, Embedded, Oracle, UI, C++, Backend; 9/20 have Primary Keyword C++.
- **hyperbolic + gen + cos + depth**: the first 20 fallback CVs come via Linux, Data, Git, Docker; 11/20 have Primary Keyword C++.
- **cosine**: the first 20 fallback CVs come via LVM, JVM, nvm, VM; 2/20 have Primary Keyword C++.

The graph's edges out of LLVM: C++ (0.95), Clang (0.55). Cosine reads names only, so its fallback follows spelling. A hyperbolic distance is symmetric, so central hubs are close to everything, and its depth term only acts in down mode. There is no ground truth for up-mode fallbacks, so this part is an eyeball check, not a measurement.


### "Looking for the C++ experts" (down)

Hundreds of test CVs say C++. Search lists them first; the interesting part is the fallback, the CVs that never say C++.

In [14]:
for m in ("graph", "hyp", "cos"):
    show("Looking for the C++ experts", "down", m, named=3, implied=10, expect="C++")

#,why,title,Primary Keyword,where
1,names C++,C++ Developer,C++ ✓,"…C++ Developer Programing skills: • C/C++(11,14,17) • STL – standard data structures and significant amount of algorithms • STL/boost/POSIX/OpenMP/MPI parallel programming • OO…"
2,names C++,Разработчик C++,C++ ✓,"…Разработчик C++ C++11/14, Boost.Asio, Boost.Accumulate, make/cmake, knowledge of modern c++ idioms like sfinae, members detector... Python, django (with sertificate), pandas / n…"
3,names C++,C++ Developer,C++ ✓,"…C++ Developer C/C++, Linux, Mac Os X, Python, STL, Qt, Network Programming, Web, Subversion, Git, POSIX, Boost.Asio, Design Patterns, UML, C++11, experience in Linux and Mac system…"
4,via OpenCL,Unity3D Developer,Unity,"…objects based on the ray-tracing algorithm, written using the SDL and OpenCL libraries. - Other graphic projects using SDL and MLX libraries. …"
5,via CMake,C Linux Embedded developer,C++ ✓,"…ux, Embedded C, Linux Kernel, ip utils/iptables, Python, DBus, yocto, CMake, netlink, bash, L2/L3 networks, OpenWRT, Docker, buildroot, mac80211 Developed a highly portable softwar…"
6,via Qt framework,Junior/Trainee developer,C++ ✓,…udio player which offers a wide range of features and is based on the Qt framework and SDL and FFmpeg libraries. The project is explained in more detail in 'achievements' section o…
7,via libcurl,Senior Software Engineer,Python,"…ostly work with Linux/Docker/Flask/Redis from backend side and C/glib/libcurl/winapi on client side. Planned/designed and developed cool sdk program, which happened to be a base fo…"
8,via Qt,Lead/Senior Embedded developer,C++ ✓,"… In a year I entered lead position. Then I decided to try myself in Qt development. And now my role is BSP engineer. I port programs on C, Qt ant Python, as well as create BSP u…"
9,via CMake,Software/Firmware Engineer | Embedded Network,C++ ✓,…ubleshooting and bug fixing • Platform optimization. • Make/CMake build infrastructure. • Release activities management • DevOps experience • SONiC exper…
10,via OpenMP,DevOps (HPC Systems Engineer),DevOps,"…mfortable with configuring, installing, and troubleshooting Open MPI, OpenMP. • Built, installed, and supported scientific software (e.g., WRF, MM5). • Knowledge of programming l…"


#,why,title,Primary Keyword,where
1,names C++,Middle embedded developer,C++ ✓,"…per 2014-2018 I was working as web dev... 2018- now Embedded C/C++ developer -company “Help Micro”, Kyiv, Ukraine; Hard skills: -C/C++ for microcontrollers; -MCUs, …"
2,names C++,Android Developer,Android,"…ity Łódź(Computer Science). In the university I worked with: Java, C/C++, SQL, Unity, C# and OOP.…"
3,names C++,C/C++ developer,C++ ✓,…C/C++ developer Linux able to manage team of 3-5 people developing modules from scratch ( starting from requirements )…
4,via C,Embedded Hardware Engineer,C++ ✓,"…Embedded Hardware Engineer I am an experienced embedded systems engineer with over 8 years of experience. I have more than 20 completed projects of varying complexity, most of whic…"
5,via C,Android developer,Android,"…Android developer Android: Open Source Project(AOSP) based infotainment car system App for football fans. Java8, Kotlin, RxJava, RxBinding, Coroutines, Live Data, ViewModel, M…"
6,via C,Embedded C Developer,C++ ✓,…Embedded C Developer Embedded Linux Developer at Inango Apr 2018 – Jun 2020 Yocto & Embedded Linux team in Intel CHD. Developing solutions for cable modem project. Focused ma…
7,via C,Embedded engineer,C++ ✓,"…Embedded engineer 5 years embedded development experience in multiple domains such as HVAC, power electronics. Experienced developer with strong C and OOP knowledge, PCB and hardwa…"
8,via C,C Linux Embedded developer,C++ ✓,"…C Linux Embedded developer I’m a developer with a deep understanding of the Linux ecosystem, from digital electronics to userspace software. My strengths are discipline and commitm…"
9,via C,Программист,C++ ✓,"…Программист Assembler Intel 51, Z80, x86, 68xxx, C, TMT-Pascal, Delphi, Pascal, Visual Basic, Altium Designer, Quartus-9, Keil, Iar - True hole procedures. - Low Power structure o…"
10,via C,Embedded C developer,C++ ✓,"…Embedded C developer C, Windows drivers (WDF, WDM), Linux drivers, USB 3.0 device drivers, PCIe device drivers. USB device drivers development (Windows, Linux), PCIe device driver…"


#,why,title,Primary Keyword,where
1,names C++,C++ Developer,C++ ✓,"…C++ Developer Carry out research and development works in sphere of ecosystems modeling (ecology, applied biotechnology) C/C+ (OOP principles knowledge, SOLID, code design patte…"
2,names C++,C++ Developer,C++ ✓,…C++ Developer I am a c ++ software engineer with 9 months of commercial development experience. Projects and technologies: Trainee project 1: • C++ application for reading and w…
3,names C++,iOS Developer,iOS,"…und in computer science, algorithms, data structures, low level and C/C++ programming, SQL. Also dealt with Android and HTML/JS projects. 1. Concrete startup success story - team …"
4,via Objective-C++,iOS Developer / Team Lead,iOS,"…IBLinter, Fastlane, Jenkins, Sourcery Languages: Swift, Objective-C, Objective-C++ Platforms: iOS I have experience working on apps in a team and alone. Developed apps from scrat…"
5,via Computer Science,Student,.NET,…Student I am a 2nd year student majoring in Computer Science. One of the projects was the development of a utility bill calculation program using the C# programming language using …
6,via Computer Science,Junior C# .Net Developer,.NET,…nnamelnikovaua.dp.ua Certificate of “HarvardX: CS50's Introduction to Computer Science”…
7,via Computer Science,Java Software Engineer,Java,…ian competition of research papers among high school students) in the Computer Science department.…
8,via Computer Science,Full Stack Developer,Node.js,… database servers. I have a BSc in Software Engineering and MSc in Computer Science. I am a fast learner and I am good at finding solution to problems. I have good teamwork a…
9,via Computer Science,Lead Java Developer,Java,…unicating with client/other teams - conducting technical interviews Computer Science Master Degree OCA Java SE 8 Programmer Contributing to Open Source and Stack Overflow Dou/…
10,via Computer Science,Software Engineer,Java,"…t in English (C2), Russian and Kazakh. With a Bachelor's degree in Computer Science from International Information Technology University (IITU), I have worked for top companies …"


In [15]:
prec = {}
for m in S:
    _, rest = search.query("Looking for the C++ experts", "down", m, k=20, names=False)
    prec[NAMES[m]] = np.mean([r["cv"]["pk"] == "C++" for r in rest])
base = np.mean(pk[~test.names(T["C++"][1]).numpy()] == "C++")
display(Markdown(f"**Share of C++ people in the first 20 fallback CVs**, against {base:.1%} among all CVs that don't say C++: " +
                 ", ".join(f"{k} {v:.0%}" for k, v in prec.items())))

**Share of C++ people in the first 20 fallback CVs**, against 1.3% among all CVs that don't say C++: cosine 0%, graph lookup 30%, cos + gen 0%, order + gen + cos 15%, box + gen + cos 15%, hyperbolic + gen + cos + depth 85%

## 7. What this shows, and what it doesn't

- **The whole pipeline needs only CVs.** Mined keywords, a co-occurrence graph and graph lookup take keyword search from chance level to a usable fallback on people who never wrote the keyword. Nothing in it is trained or labelled.
- **On this collection the graph is the method.** The embedding heads learn from the graph and don't beat it, because every test keyword is in the same closed vocabulary. Their potential case, keywords the graph has no edge for, is untested.
- **"Experts" and "would be nice" aren't modelled.** Search finds who has a skill, not how well or how much it's wanted. Within the CVs that name C++, the order comes only from how many of their other keywords support it.
- **The truth is coarse.** One self-picked Primary Keyword per CV, for 18 technologies. It says nothing about LLVM → C++ as such. Up-mode fallbacks are checked only on 22 hand-picked requests.
- **The vocabulary is noisy** (see the long tail and the random edges in sections 2 and 3). With a lift of 3 and 5 co-occurrences required, most of that noise never forms an edge.
- **Not run:** the NLI edge filter (`verify` with `nli_judge`), seeds beyond 0, and serving from Qdrant.

## 8. The demo page

`cvsearch.demo.export` writes the search as one self-contained HTML page (`data/djinni/cvsearch_demo.html`, from the template `tools/cvsearch_demo.html`). A static page can't run the models, so it stores, for the 1,000 keywords most named in the test CVs plus the demo ones, the top CVs per mode and method from the scorers above. It parses free text with the same keyword rules, in JavaScript.

In [16]:
from pathlib import Path
from cvsearch.demo import export
page = Path("data/djinni/cvsearch_demo.html")
print(export(page, test, graph, {k: S[k] for k in ("graph", "hyp", "box", "order", "cos")}, NAMES, results), "→", page)
print(f"notebook ran in {(time.time() - t0) / 60:.0f} minutes")

{'queries': 1001, 'cvs': 5919, 'MB': 9.2} → data/djinni/cvsearch_demo.html
notebook ran in 16 minutes
